
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.


# Supervision / Feature Ablation
## Does the gain come from context/architecture, or from correct future supervision?

This notebook isolates observable-context effects from correct future supervision.

### Methods
1. **Pattern** — pattern similarity only.
2. **Pattern + Handcrafted Context** — same observable 7-D context features, but no MLP and no future supervision. Pattern rank and context-similarity rank are fused 50/50 with no tuned parameter.
3. **Shuffled-Future MLP** — same MLP, same features, same candidate pool and same training recipe as the proposed method, but the query future is cyclically shuffled within channel.
4. **Future-Supervised MLP (Ours)** — same architecture/features as (3), with the correct future-compatibility target.

Critical comparisons:

\[
\text{Pattern}\rightarrow\text{Pattern+Context}
\]

tests whether the extra observable context alone explains the gain, while

\[
\text{Shuffled-Future MLP}\rightarrow\text{Future-Supervised MLP}
\]

holds architecture and inputs fixed and isolates the value of correct future supervision.

### Frozen protocol
- Electricity, Traffic, Exchange, Solar
- \(L=96,\; H\in\{24,48,96\}\)
- Same-channel Pattern Top-\(M\), \(M=100,\;K=10\)
- Fixed train-scale target
- Existing Learned/Shuffled results averaged over 5 seeds
- Moving-block bootstrap, 5,000 replicates

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


## 0. Imports and paths

In [ ]:

from pathlib import Path
import math
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RESULT_DIR = REPO_WORK_ROOT / "final_confirmatory"
CACHE_DIR = RESULT_DIR / "cache"
OUT_DIR = RESULT_DIR / "ablation_supervision_features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["Electricity", "Traffic", "Exchange", "Solar"]
HORIZONS = [24, 48, 96]

TOP_M = 100
TOP_K = 10
MAX_MEMORY_WINDOWS = 50000
BLOCK_ANCHORS = 10
N_BOOT = 5000

DATASET_SEED = {
    "Electricity": 3303,
    "Traffic": 4404,
    "Exchange": 5505,
    "Solar": 6606,
}

print("Input:", RESULT_DIR)
print("Output:", OUT_DIR)
assert RESULT_DIR.exists()
assert CACHE_DIR.exists()


## 1. Check prerequisites

In [ ]:

required = [
    RESULT_DIR / "00_data_manifest.csv",
    RESULT_DIR / "09_main_confirmatory_summary.csv",
]

for d in DATASETS:
    for H in HORIZONS:
        required += [
            CACHE_DIR / f"{d}_H{H}_windows.npz",
            CACHE_DIR / f"{d}_H{H}_meta.csv.gz",
            CACHE_DIR / f"{d}_H{H}_same_topM.npz",
            RESULT_DIR / f"query_level_{d}_H{H}.csv.gz",
        ]

missing = [str(p) for p in required if not p.exists()]
if missing:
    print("\n".join(missing))
    raise FileNotFoundError(
        "Run the final confirmatory notebook first."
    )

print("All prerequisite files found.")



## 2. Reconstruct the exact context scaling

The confirmatory experiment fits a median/IQR scaler only on the sampled train-memory contexts.  
The following cells reconstruct the same temporal split and deterministic channel-balanced sampling.


In [ ]:

manifest = pd.read_csv(RESULT_DIR / "00_data_manifest.csv")

N_ROWS = {
    str(r["Dataset"]): int(r["Rows"])
    for _, r in manifest.iterrows()
    if str(r["Dataset"]) in DATASETS
}

def split_boundaries(n):
    train_end = int(0.70 * n)
    return {
        "train_end": train_end,
        "val_end": int(0.80 * n),
        "inner_memory_end": int(0.60 * train_end),
    }

SPLITS = {d: split_boundaries(N_ROWS[d]) for d in DATASETS}

def balanced_subset(meta, indices, max_n, seed):
    indices = np.asarray(indices, dtype=np.int64)
    if len(indices) <= max_n:
        return np.sort(indices)

    rng = np.random.default_rng(seed)
    channels = meta.iloc[indices]["ChannelIndex"].to_numpy(dtype=np.int64)
    unique_channels = np.unique(channels)
    random_channel_order = rng.permutation(unique_channels)

    base = max_n // len(unique_channels)
    extra = max_n % len(unique_channels)
    chosen_parts = []

    for rank, c in enumerate(random_channel_order):
        pos = indices[channels == c]
        quota = base + (1 if rank < extra else 0)
        take = min(quota, len(pos))
        if take > 0:
            chosen_parts.append(
                rng.choice(pos, size=take, replace=False)
            )

    chosen = np.unique(np.concatenate(chosen_parts))

    if len(chosen) < max_n:
        remaining = np.setdiff1d(indices, chosen, assume_unique=False)
        add_n = min(max_n - len(chosen), len(remaining))
        if add_n > 0:
            chosen = np.concatenate([
                chosen,
                rng.choice(remaining, size=add_n, replace=False),
            ])

    return np.sort(chosen.astype(np.int64))

def fit_robust_scaler(x):
    med = np.median(x, axis=0)
    q25 = np.percentile(x, 25, axis=0)
    q75 = np.percentile(x, 75, axis=0)
    iqr = q75 - q25
    iqr = np.where(iqr < 1e-5, 1.0, iqr)
    return med.astype(np.float32), iqr.astype(np.float32)

def apply_robust_scaler(x, med, iqr):
    z = (x - med) / iqr
    return np.clip(z, -8.0, 8.0).astype(np.float32)


In [ ]:

TASK = {}

for d in DATASETS:
    for H in HORIZONS:
        key = (d, H)

        w = np.load(CACHE_DIR / f"{d}_H{H}_windows.npz")
        meta = pd.read_csv(CACHE_DIR / f"{d}_H{H}_meta.csv.gz")
        p = np.load(CACHE_DIR / f"{d}_H{H}_same_topM.npz")

        context = w["context"].astype(np.float32)
        future = w["future"].astype(np.float32)

        future_end = meta["FutureEnd"].to_numpy(dtype=np.int64)
        full_train_memory = np.where(
            future_end < SPLITS[d]["inner_memory_end"]
        )[0]

        base = DATASET_SEED[d] + H * 10
        train_memory = balanced_subset(
            meta,
            full_train_memory,
            MAX_MEMORY_WINDOWS,
            base + 1,
        )

        med, iqr = fit_robust_scaler(context[train_memory])
        context_scaled = apply_robust_scaler(context, med, iqr)

        q_idx = p["test_query"].astype(np.int64)
        cand_idx = p["test_idx"].astype(np.int64)

        TASK[key] = {
            "pattern_score": p["test_score"].astype(np.float32),
            "q_context": context_scaled[q_idx],
            "cand_context": context_scaled[cand_idx],
            "q_future": future[q_idx],
            "cand_future": future[cand_idx],
            "anchor": meta.iloc[q_idx]["Anchor"].to_numpy(dtype=np.int64),
            "channel": meta.iloc[q_idx]["ChannelIndex"].to_numpy(dtype=np.int64),
        }

        assert TASK[key]["pattern_score"].shape[1] == TOP_M

        print(
            d, H,
            "| test queries:", len(q_idx),
            "| train-memory scaler:", len(train_memory)
        )


## 3. Metrics

In [ ]:

def future_distance(q_future, cand_future):
    return np.mean(
        (cand_future - q_future[:, None, :]) ** 2,
        axis=2,
    )

def topk_from_scores(score, k):
    idx = np.argpartition(-score, kth=k-1, axis=1)[:, :k]
    row = np.arange(len(score))[:, None]
    order = np.argsort(-score[row, idx], axis=1)
    return idx[row, order]

def gather2(x, idx):
    row = np.arange(len(x))[:, None]
    return x[row, idx]

def gather3(x, idx):
    row = np.arange(len(x))[:, None]
    return x[row, idx, :]

def query_metrics(score, phase):
    fdist = future_distance(
        phase["q_future"],
        phase["cand_future"],
    )

    selected = topk_from_scores(score, TOP_K)

    analog = gather2(
        fdist,
        selected,
    ).mean(axis=1)

    pred = gather3(
        phase["cand_future"],
        selected,
    ).mean(axis=1)

    forecast = np.mean(
        (pred - phase["q_future"]) ** 2,
        axis=1,
    )

    return pd.DataFrame({
        "AnalogFutureMSE": analog.astype(np.float32),
        "RetrievalForecastMSE": forecast.astype(np.float32),
    })



## 4. Parameter-free Pattern + Handcrafted Context

Context score:

\[
s_{\rm ctx}(q,i)
=
-\frac{1}{D}\|z_q-z_i\|_2^2.
\]

To avoid tuning a fusion coefficient with future labels, both pattern and context scores are converted to per-query normalized ranks and fused equally:

\[
s_{\rm P+C}
=
\frac{1}{2}r_{\rm pattern}
+
\frac{1}{2}r_{\rm context}.
\]

This baseline has access to the same observable context features but has **no learned parameter and no future supervision**.


In [ ]:

def row_rank_score(score):
    n, m = score.shape
    order = np.argsort(score, axis=1)
    ranks = np.empty_like(order, dtype=np.float32)
    row = np.arange(n)[:, None]
    ranks[row, order] = np.arange(m, dtype=np.float32)[None, :]
    if m > 1:
        ranks /= float(m - 1)
    return ranks

def handcrafted_scores(phase):
    diff = (
        phase["q_context"][:, None, :]
        - phase["cand_context"]
    )

    context_score = -np.mean(
        diff ** 2,
        axis=2,
    ).astype(np.float32)

    pattern_rank = row_rank_score(
        phase["pattern_score"]
    )
    context_rank = row_rank_score(
        context_score
    )

    fused_score = (
        0.5 * pattern_rank
        + 0.5 * context_rank
    ).astype(np.float32)

    return context_score, fused_score


## 5. Evaluate all four ablation methods

In [ ]:

QUERY = {}
summary_rows = []

for d in DATASETS:
    for H in HORIZONS:
        key = (d, H)
        phase = TASK[key]

        pattern_m = query_metrics(
            phase["pattern_score"],
            phase,
        )

        context_score, fusion_score = handcrafted_scores(phase)

        context_m = query_metrics(context_score, phase)
        fusion_m = query_metrics(fusion_score, phase)

        existing = pd.read_csv(
            RESULT_DIR / f"query_level_{d}_H{H}.csv.gz"
        )

        assert len(existing) == len(pattern_m)

        # Sanity check: exact Pattern reproduction.
        max_diff = float(np.max(np.abs(
            pattern_m["AnalogFutureMSE"].to_numpy()
            - existing["Pattern_AnalogFutureMSE"].to_numpy()
        )))
        assert max_diff < 1e-5, (d, H, max_diff)

        q = pd.DataFrame({
            "Anchor": phase["anchor"],
            "ChannelIndex": phase["channel"],

            "Pattern_AnalogFutureMSE":
                pattern_m["AnalogFutureMSE"].to_numpy(),
            "ContextOnly_AnalogFutureMSE":
                context_m["AnalogFutureMSE"].to_numpy(),
            "PatternContext_AnalogFutureMSE":
                fusion_m["AnalogFutureMSE"].to_numpy(),
            "ShuffledFuture_AnalogFutureMSE":
                existing["ShuffledFuture_AnalogFutureMSE"].to_numpy(),
            "Learned_AnalogFutureMSE":
                existing["Learned_AnalogFutureMSE"].to_numpy(),

            "Pattern_RetrievalForecastMSE":
                pattern_m["RetrievalForecastMSE"].to_numpy(),
            "ContextOnly_RetrievalForecastMSE":
                context_m["RetrievalForecastMSE"].to_numpy(),
            "PatternContext_RetrievalForecastMSE":
                fusion_m["RetrievalForecastMSE"].to_numpy(),
            "ShuffledFuture_RetrievalForecastMSE":
                existing["ShuffledFuture_RetrievalForecastMSE"].to_numpy(),
            "Learned_RetrievalForecastMSE":
                existing["Learned_RetrievalForecastMSE"].to_numpy(),
        })

        QUERY[key] = q

        for method in [
            "Pattern",
            "ContextOnly",
            "PatternContext",
            "ShuffledFuture",
            "Learned",
        ]:
            summary_rows.append({
                "Dataset": d,
                "Horizon": H,
                "Method": method,
                "AnalogFutureMSE": float(
                    q[f"{method}_AnalogFutureMSE"].mean()
                ),
                "RetrievalForecastMSE": float(
                    q[f"{method}_RetrievalForecastMSE"].mean()
                ),
            })

        q.to_csv(
            OUT_DIR / f"query_level_{d}_H{H}.csv.gz",
            index=False,
            compression="gzip",
        )

summary = pd.DataFrame(summary_rows)
display(summary.sort_values(
    ["Dataset", "Horizon", "AnalogFutureMSE"]
))

summary.to_csv(
    OUT_DIR / "01_ablation_summary.csv",
    index=False,
)


## 6. Effect decomposition

In [ ]:

relative_rows = []

for d in DATASETS:
    for H in HORIZONS:
        x = summary[
            (summary["Dataset"] == d)
            & (summary["Horizon"] == H)
        ].set_index("Method")

        def v(m):
            return float(x.loc[m, "AnalogFutureMSE"])

        pattern = v("Pattern")
        fusion = v("PatternContext")
        shuffled = v("ShuffledFuture")
        learned = v("Learned")

        relative_rows.append({
            "Dataset": d,
            "Horizon": H,
            "Pattern": pattern,
            "PatternContext": fusion,
            "ShuffledFuture": shuffled,
            "Learned": learned,

            # Extra observable context, no learned/future target.
            "Pattern_to_ContextFusion_%":
                100.0 * (pattern - fusion) / pattern,

            # Same architecture/features; only correct future correspondence differs.
            "Shuffled_to_Learned_%":
                100.0 * (shuffled - learned) / shuffled,

            # Does future-supervised learning beat deterministic context use?
            "ContextFusion_to_Learned_%":
                100.0 * (fusion - learned) / fusion,
        })

relative = pd.DataFrame(relative_rows)
display(relative)

relative.to_csv(
    OUT_DIR / "02_effect_decomposition.csv",
    index=False,
)


## 7. Moving-block bootstrap

In [ ]:

def moving_block_bootstrap(x, block_len, n_boot, seed):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    assert n >= block_len

    rng = np.random.default_rng(seed)
    n_blocks = math.ceil(n / block_len)
    max_start = n - block_len

    means = np.empty(n_boot, dtype=np.float64)

    for b in range(n_boot):
        parts = []
        for _ in range(n_blocks):
            start = int(rng.integers(0, max_start + 1))
            parts.append(x[start:start + block_len])
        sample = np.concatenate(parts)[:n]
        means[b] = sample.mean()

    lo = float(np.quantile(means, 0.025))
    hi = float(np.quantile(means, 0.975))

    return {
        "ObservedImprovement": float(x.mean()),
        "CI_2.5%": lo,
        "CI_97.5%": hi,
        "SignificantImprovement": bool(lo > 0),
    }

comparisons = [
    ("Pattern", "PatternContext"),
    ("ShuffledFuture", "Learned"),
    ("PatternContext", "Learned"),
    ("Pattern", "Learned"),
]

bootstrap_rows = []

for d in DATASETS:
    for H in HORIZONS:
        q = QUERY[(d, H)]

        for baseline, proposed in comparisons:
            diff = (
                q[f"{baseline}_AnalogFutureMSE"]
                - q[f"{proposed}_AnalogFutureMSE"]
            )

            anchor_diff = (
                pd.DataFrame({
                    "Anchor": q["Anchor"],
                    "Diff": diff,
                })
                .groupby("Anchor")["Diff"]
                .mean()
                .sort_index()
                .to_numpy()
            )

            r = moving_block_bootstrap(
                anchor_diff,
                BLOCK_ANCHORS,
                N_BOOT,
                seed=(
                    DATASET_SEED[d]
                    + H * 100
                    + len(bootstrap_rows)
                ),
            )

            r.update({
                "Dataset": d,
                "Horizon": H,
                "Baseline": baseline,
                "Proposed": proposed,
            })
            bootstrap_rows.append(r)

bootstrap = pd.DataFrame(bootstrap_rows)
display(bootstrap)

bootstrap.to_csv(
    OUT_DIR / "03_moving_block_bootstrap.csv",
    index=False,
)


## 8. Paper-ready table

In [ ]:

def sig(d, H, baseline, proposed):
    x = bootstrap[
        (bootstrap["Dataset"] == d)
        & (bootstrap["Horizon"] == H)
        & (bootstrap["Baseline"] == baseline)
        & (bootstrap["Proposed"] == proposed)
    ]
    assert len(x) == 1
    return bool(x["SignificantImprovement"].iloc[0])

paper_rows = []

for _, r in relative.iterrows():
    d = r["Dataset"]
    H = int(r["Horizon"])

    paper_rows.append({
        "Dataset": d,
        "Horizon": H,
        "Pattern": r["Pattern"],
        "Pattern+Context": r["PatternContext"],
        "Shuffled-MLP": r["ShuffledFuture"],
        "Future-Supervised-MLP": r["Learned"],
        "ContextGain_%": r["Pattern_to_ContextFusion_%"],
        "FutureSupervisionGain_%": r["Shuffled_to_Learned_%"],
        "ContextGain_Sig": sig(
            d, H, "Pattern", "PatternContext"
        ),
        "FutureSupervisionGain_Sig": sig(
            d, H, "ShuffledFuture", "Learned"
        ),
        "LearnedBeatsContext_Sig": sig(
            d, H, "PatternContext", "Learned"
        ),
    })

paper_table = pd.DataFrame(paper_rows)
display(paper_table)

paper_table.to_csv(
    OUT_DIR / "04_paper_task_table.csv",
    index=False,
)


## 9. Dataset-level summary

In [ ]:

dataset_rows = []

for d in DATASETS:
    x = relative[relative["Dataset"] == d]
    b = bootstrap[bootstrap["Dataset"] == d]

    def sig_count(base, prop):
        return int(
            b[
                (b["Baseline"] == base)
                & (b["Proposed"] == prop)
            ]["SignificantImprovement"].sum()
        )

    dataset_rows.append({
        "Dataset": d,
        "Mean_ContextGain_%": float(
            x["Pattern_to_ContextFusion_%"].mean()
        ),
        "ContextGain_Sig_Horizons": sig_count(
            "Pattern", "PatternContext"
        ),
        "Mean_FutureSupervisionGain_%": float(
            x["Shuffled_to_Learned_%"].mean()
        ),
        "FutureSupervisionGain_Sig_Horizons": sig_count(
            "ShuffledFuture", "Learned"
        ),
        "Mean_ContextFusion_to_Learned_%": float(
            x["ContextFusion_to_Learned_%"].mean()
        ),
        "LearnedBeatsContext_Sig_Horizons": sig_count(
            "PatternContext", "Learned"
        ),
    })

dataset_summary = pd.DataFrame(dataset_rows)
display(dataset_summary)

dataset_summary.to_csv(
    OUT_DIR / "05_dataset_summary.csv",
    index=False,
)


## 10. Automatic supervision check

In [ ]:

n_tasks = len(relative)

check = pd.DataFrame([{
    "Tasks": n_tasks,

    "PatternContextBeatsPattern": int(
        (relative["PatternContext"] < relative["Pattern"]).sum()
    ),

    "PatternContextSignificant": int(
        bootstrap[
            (bootstrap["Baseline"] == "Pattern")
            & (bootstrap["Proposed"] == "PatternContext")
        ]["SignificantImprovement"].sum()
    ),

    "LearnedBeatsShuffledMLP": int(
        (relative["Learned"] < relative["ShuffledFuture"]).sum()
    ),

    "FutureSupervisionSignificant": int(
        bootstrap[
            (bootstrap["Baseline"] == "ShuffledFuture")
            & (bootstrap["Proposed"] == "Learned")
        ]["SignificantImprovement"].sum()
    ),

    "LearnedBeatsPatternContext": int(
        (relative["Learned"] < relative["PatternContext"]).sum()
    ),

    "LearnedOverContextSignificant": int(
        bootstrap[
            (bootstrap["Baseline"] == "PatternContext")
            & (bootstrap["Proposed"] == "Learned")
        ]["SignificantImprovement"].sum()
    ),
}])

display(check)

check.to_csv(
    OUT_DIR / "06_supervision_diagnostic_check.csv",
    index=False,
)



# Interpretation

Read the result in this order.

**1. Pattern \(\rightarrow\) Pattern+Context**  
If this improves, observable context is itself useful. This does not invalidate the method; it tells us how much can be obtained without learning or future supervision.

**2. Shuffled-Future MLP \(\rightarrow\) Future-Supervised MLP**  
This is the cleanest architecture-matched comparison. Both methods use the same MLP and the same features; only the correctness of the training-time future correspondence differs.

**3. Pattern+Context \(\rightarrow\) Future-Supervised MLP**  
If Learned remains better, the result shows that the gain is not explained by a fixed handcrafted use of the extra context features.

Suggested paper wording if the evidence is strong:

> Adding observable context through a parameter-free rank fusion does not explain the proposed model's gains. More importantly, with architecture and input features held fixed, replacing shuffled futures with the correct training-time future correspondence consistently improves retrieval quality, isolating future-compatible supervision as the key source of the gain.

Do **not** retune the proposed model based on this result. The next experiment should be the separately designed similarity-robustness study.
